# Part 5 — Model Comparison & Hyperparameter Tuning

**Goal:** Go beyond the RF baseline from Part 4.
Tune RF, XGBoost and SVR with Optuna, then add a PyTorch Lightning MLP
as a first step toward the GNN benchmarked in DeepEGFR (2025).

## What this notebook adds

| Model | New in Part 5 | Why |
|---|---|---|
| RF (tuned) | Optuna search over depth/estimators | Honest improvement over Part 4 defaults |
| **XGBoost** | Full model + Optuna | Often beats RF on tabular QSAR data |
| **SVR (RBF)** | Full model + Optuna | Literature SOTA for ECFP regression (Vignaux 2023: R²=0.81) |
| **MLP (Lightning)** | PyTorch Lightning | GPU-compatible; bridge to GNN extension |

## Critical design rules from Part 4 (unchanged)
- **Same Butina cluster IDs** loaded from `.npy` — no re-clustering
- **SVR uses Pipeline(scaler + SVR)** so StandardScaler fits only on train data inside each fold
- **`class_weight='balanced'`** for all classifiers
- **No evaluation on training data** — all numbers are out-of-fold only

**Inputs:** `X_ecfp6.csv`, `y_pchembl.csv`, `y_binary.csv`,
`butina_cluster_ids_reg.npy`, `butina_cluster_ids_clf.npy`

**Outputs:** `part5_final_comparison.csv`, `best_model_part5.pkl`

---
## 0. Installs & Imports

In [ ]:
# Uncomment on first Colab run:
# !pip install rdkit-pypi optuna xgboost lightning torchmetrics

import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    roc_auc_score, balanced_accuracy_score,
    matthews_corrcoef, f1_score
)
from sklearn.model_selection import BaseCrossValidator
from scipy.stats import pearsonr

# XGBoost
import xgboost as xgb

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# PyTorch + Lightning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import torchmetrics

import matplotlib.pyplot as plt

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
print(f'PyTorch  : {torch.__version__}')
print(f'Lightning: {L.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

---
## 1. Load Data & Cluster IDs

In [ ]:
# ── Feature matrix (ECFP6 — best in Part 4 ablation) ─────────────────────────
X_df    = pd.read_csv('X_ecfp6.csv')
mol_ids = X_df['molecule_chembl_id']
X       = X_df.drop(columns='molecule_chembl_id').values.astype(np.float32)

# ── Targets ───────────────────────────────────────────────────────────────────
y_pchembl_df = pd.read_csv('y_pchembl.csv')
y_binary_df  = pd.read_csv('y_binary.csv')

y_reg = (y_pchembl_df
         .set_index('molecule_chembl_id')
         .loc[mol_ids, 'pchembl_value']
         .values.astype(np.float32))

label_map   = {'active': 1, 'inactive': 0}
clf_mask    = mol_ids.isin(y_binary_df['molecule_chembl_id']).values
X_clf       = X[clf_mask]
mol_ids_clf = mol_ids[clf_mask].reset_index(drop=True)
y_clf       = (y_binary_df
               .set_index('molecule_chembl_id')
               .loc[mol_ids_clf, 'bioactivity_class']
               .map(label_map)
               .values)

# ── Butina cluster IDs from Part 4 (do NOT re-cluster here) ──────────────────
cluster_ids_reg = np.load('butina_cluster_ids_reg.npy')
cluster_ids_clf = np.load('butina_cluster_ids_clf.npy')

print(f'X (regression)      : {X.shape}')
print(f'X (classification)  : {X_clf.shape}')
print(f'y_reg range         : [{y_reg.min():.2f}, {y_reg.max():.2f}]')
print(f'Class balance       : active={y_clf.sum()}, inactive={(y_clf==0).sum()}')

---
## 2. CV Utilities (identical to Part 4)

In [ ]:
class ButinaCrossValidator(BaseCrossValidator):
    def __init__(self, n_splits=5):
        self.n_splits = n_splits

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def _iter_test_masks(self, X=None, y=None, groups=None):
        if groups is None:
            raise ValueError('Provide groups=cluster_ids')
        unique_cids = np.unique(groups)
        sizes       = {c: np.sum(groups == c) for c in unique_cids}
        sorted_cids = sorted(unique_cids, key=lambda c: -sizes[c])
        fold_assign = {c: i % self.n_splits for i, c in enumerate(sorted_cids)}
        for fold in range(self.n_splits):
            yield np.array([fold_assign[c] == fold for c in groups], dtype=bool)


def regression_metrics(y_true, y_pred):
    r, _ = pearsonr(y_true, y_pred)
    return {'R2': r2_score(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE': mean_absolute_error(y_true, y_pred),
            'Pearson_r': r}

def classification_metrics(y_true, y_pred, y_prob=None):
    return {'AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
            'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
            'MCC': matthews_corrcoef(y_true, y_pred),
            'MacroF1': f1_score(y_true, y_pred, average='macro')}

def run_cv_regression(model, X, y, cluster_ids, n_splits=5):
    cv, rows = ButinaCrossValidator(n_splits), []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups=cluster_ids), 1):
        model.fit(X[tr], y[tr])
        m = regression_metrics(y[te], model.predict(X[te]))
        m['Fold'] = fold
        rows.append(m)
    df = pd.DataFrame(rows).set_index('Fold')
    return pd.concat([df,
                      df.mean().rename('mean').to_frame().T,
                      df.std().rename('std').to_frame().T])

def run_cv_classification(model, X, y, cluster_ids, n_splits=5):
    cv, rows = ButinaCrossValidator(n_splits), []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups=cluster_ids), 1):
        model.fit(X[tr], y[tr])
        prob = model.predict_proba(X[te])[:, 1]
        m = classification_metrics(y[te], model.predict(X[te]), prob)
        m['Fold'] = fold
        rows.append(m)
    df = pd.DataFrame(rows).set_index('Fold')
    return pd.concat([df,
                      df.mean().rename('mean').to_frame().T,
                      df.std().rename('std').to_frame().T])

print('CV utilities ready.')

---
## 3. Optuna Tuning — RF, XGBoost, SVR

**Inner vs outer CV — why this matters:**
Optuna tunes hyperparameters on the **training portion of fold 1** only.
The resulting best params are then evaluated across all 5 folds.
This prevents the tuning process from leaking information from the test sets.

**SVR requires a Pipeline:**
SVR with RBF kernel is sensitive to feature scale.
`Pipeline([StandardScaler, SVR])` ensures the scaler is always fitted
only on training data within each fold — never on the test fold.

In [ ]:
# ── Extract fold-1 training data for the inner tuning loop ───────────────────
cv_outer    = ButinaCrossValidator(n_splits=5)
outer_splits = list(cv_outer.split(X, y_reg, groups=cluster_ids_reg))
tune_tr, _  = outer_splits[0]
X_tune, y_tune, ids_tune = X[tune_tr], y_reg[tune_tr], cluster_ids_reg[tune_tr]

# Inner CV for fast evaluation inside Optuna
cv_inner      = ButinaCrossValidator(n_splits=3)
inner_splits  = list(cv_inner.split(X_tune, y_tune, groups=ids_tune))
in_tr, in_val = inner_splits[0]

N_TRIALS  = 50   # increase to 100+ for production
best_params = {}

for model_name in ['RF', 'XGB', 'SVR']:
    def objective(trial, name=model_name):
        if name == 'RF':
            m = RandomForestRegressor(
                n_estimators    = trial.suggest_int('n_estimators', 100, 800),
                max_depth       = trial.suggest_int('max_depth', 5, 40),
                min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 10),
                random_state=RANDOM_STATE, n_jobs=-1)
        elif name == 'XGB':
            m = xgb.XGBRegressor(
                n_estimators     = trial.suggest_int('n_estimators', 100, 600),
                max_depth        = trial.suggest_int('max_depth', 3, 10),
                learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                subsample        = trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
                random_state=RANDOM_STATE, verbosity=0)
        elif name == 'SVR':
            m = Pipeline([('scaler', StandardScaler()),
                          ('svr', SVR(
                              C      = trial.suggest_float('C', 0.1, 100, log=True),
                              epsilon= trial.suggest_float('epsilon', 0.01, 1.0),
                              gamma  = trial.suggest_categorical('gamma', ['scale', 'auto']),
                              kernel='rbf'))])
        m.fit(X_tune[in_tr], y_tune[in_tr])
        return r2_score(y_tune[in_val], m.predict(X_tune[in_val]))

    print(f'Tuning {model_name} ({N_TRIALS} trials)...')
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params[model_name] = study.best_params
    print(f'  Best inner R² = {study.best_value:.4f}')
    print(f'  Params: {study.best_params}\n')

In [ ]:
# ── Build tuned models ────────────────────────────────────────────────────────
p = best_params

rf_tuned = RandomForestRegressor(
    n_estimators    =p['RF']['n_estimators'],
    max_depth       =p['RF']['max_depth'],
    min_samples_leaf=p['RF']['min_samples_leaf'],
    random_state=RANDOM_STATE, n_jobs=-1)

xgb_tuned = xgb.XGBRegressor(
    n_estimators    =p['XGB']['n_estimators'],
    max_depth       =p['XGB']['max_depth'],
    learning_rate   =p['XGB']['learning_rate'],
    subsample       =p['XGB']['subsample'],
    colsample_bytree=p['XGB']['colsample_bytree'],
    random_state=RANDOM_STATE, verbosity=0)

svr_tuned = Pipeline([('scaler', StandardScaler()),
                       ('svr', SVR(
                           C      =p['SVR']['C'],
                           epsilon=p['SVR']['epsilon'],
                           gamma  =p['SVR']['gamma'],
                           kernel='rbf'))])

# ── Outer 5-fold CV for all three ────────────────────────────────────────────
print('Running outer 5-fold Butina CV...')
results_reg = {}
for name, model in [('RF (tuned)', rf_tuned),
                     ('XGBoost (tuned)', xgb_tuned),
                     ('SVR (tuned)', svr_tuned)]:
    print(f'  {name}...')
    res = run_cv_regression(model, X, y_reg, cluster_ids_reg)
    results_reg[name] = res
    m, s = res.loc['mean'], res.loc['std']
    print(f'    R²={m.R2:.3f}±{s.R2:.3f}  RMSE={m.RMSE:.3f}  r={m.Pearson_r:.3f}')

---
## 4. MLP with PyTorch Lightning

### Why Lightning instead of plain PyTorch?
Lightning separates the **what** (model, loss, metrics) from the **how**
(training loop, device management, early stopping).
You write a `LightningModule` and Lightning handles:
- Moving tensors to GPU automatically (`accelerator='auto'`)
- Early stopping when `val_loss` stops improving
- Learning rate scheduling

### Architecture
```
ECFP6 input  →  BatchNorm → Linear(512) → ReLU → Dropout(0.3)
             →  BatchNorm → Linear(256) → ReLU → Dropout(0.3)
             →  BatchNorm → Linear(128) → ReLU
             →  Linear(1)   ← regression output
```
BatchNorm before each layer stabilises training on sparse binary ECFP6 inputs.
Dropout regularises against overfitting on the ~2000-compound dataset.

### Why MLP may underperform classical models here
With ~2000 compounds, SVR and XGBoost typically match or beat MLPs on tabular QSAR data.
The MLP is included as a bridge to a future GNN extension (which operates on the
molecular graph directly and can genuinely outperform fingerprint-based models).

In [ ]:
class MoleculeDataset(Dataset):
    """PyTorch Dataset wrapping numpy feature/target arrays."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):          return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class EGFRDataModule(L.LightningDataModule):
    """LightningDataModule wrapping pre-split numpy arrays."""
    def __init__(self, X_train, y_train, X_val, y_val, batch_size=64):
        super().__init__()
        self.train_ds   = MoleculeDataset(X_train, y_train)
        self.val_ds     = MoleculeDataset(X_val, y_val)
        self.batch_size = batch_size
    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,  num_workers=0)
    def val_dataloader(self):
        return DataLoader(self.val_ds,   batch_size=self.batch_size, shuffle=False, num_workers=0)


class MLPRegressor(L.LightningModule):
    """3-layer MLP for pChEMBL regression."""
    def __init__(self, input_dim, hidden_dims=(512, 256, 128), dropout=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        layers, in_dim = [], input_dim
        for h in hidden_dims:
            layers += [nn.BatchNorm1d(in_dim), nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net     = nn.Sequential(*layers)
        self.val_r2  = torchmetrics.R2Score()

    def forward(self, x): return self.net(x)

    def training_step(self, batch, _):
        x, y   = batch
        loss   = nn.functional.mse_loss(self(x), y)
        self.log('train_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, _):
        x, y  = batch
        y_hat = self(x)
        loss  = nn.functional.mse_loss(y_hat, y)
        self.val_r2(y_hat.squeeze(), y.squeeze())
        self.log('val_loss', loss,     prog_bar=True, on_epoch=True, on_step=False)
        self.log('val_r2',   self.val_r2, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self):
        opt   = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'monitor': 'val_loss'}}


print('Lightning classes defined.')

In [ ]:
# ── MLP Butina 5-fold CV ──────────────────────────────────────────────────────
input_dim        = X.shape[1]
mlp_fold_results = []

for fold, (tr, te) in enumerate(
    ButinaCrossValidator(5).split(X, y_reg, groups=cluster_ids_reg), start=1
):
    print(f'MLP Fold {fold}/5...')
    dm    = EGFRDataModule(X[tr], y_reg[tr], X[te], y_reg[te], batch_size=64)
    model = MLPRegressor(input_dim=input_dim, lr=1e-3)

    trainer = L.Trainer(
        max_epochs=100,
        accelerator='auto', devices=1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, mode='min')],
        enable_progress_bar=False,
        logger=False,
        enable_checkpointing=False,
    )
    trainer.fit(model, dm)

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.tensor(X[te], dtype=torch.float32)).squeeze().cpu().numpy()

    m = regression_metrics(y_reg[te], y_pred)
    m['Fold'] = fold
    mlp_fold_results.append(m)
    print(f'  R²={m["R2"]:.3f}  RMSE={m["RMSE"]:.3f}')

mlp_df   = pd.DataFrame(mlp_fold_results).set_index('Fold')
mlp_mean = mlp_df.mean()
mlp_std  = mlp_df.std()
print(f'\nMLP mean: R²={mlp_mean.R2:.3f}±{mlp_std.R2:.3f}')

---
## 5. Final Comparison Table

In [ ]:
rows = []
for name, res in results_reg.items():
    m, s = res.loc['mean'], res.loc['std']
    rows.append({'Model': name, 'Features': 'ECFP6', 'Split': 'Butina 5-fold',
                 'R² (mean±std)': f'{m.R2:.3f} ± {s.R2:.3f}',
                 'RMSE': f'{m.RMSE:.3f}', 'MAE': f'{m.MAE:.3f}',
                 'Pearson r': f'{m.Pearson_r:.3f}'})

rows.append({'Model': 'MLP (Lightning)', 'Features': 'ECFP6', 'Split': 'Butina 5-fold',
             'R² (mean±std)': f'{mlp_mean.R2:.3f} ± {mlp_std.R2:.3f}',
             'RMSE': f'{mlp_mean.RMSE:.3f}', 'MAE': f'{mlp_mean.MAE:.3f}',
             'Pearson r': f'{mlp_mean.Pearson_r:.3f}'})

rows.append({'Model': '── Literature (different target/dataset) ──',
             'Features': '', 'Split': '', 'R² (mean±std)': '',
             'RMSE': '', 'MAE': '', 'Pearson r': ''})

rows.append({'Model': 'SVR (Vignaux 2023, AChE)', 'Features': 'ECFP6',
             'Split': '5-fold CV', 'R² (mean±std)': '0.810',
             'RMSE': '0.730', 'MAE': '0.550', 'Pearson r': '0.760'})

comparison_df = pd.DataFrame(rows)
print('=== REGRESSION COMPARISON (Butina 5-fold CV) ===')
print(comparison_df.to_string(index=False))
comparison_df.to_csv('part5_final_comparison.csv', index=False)

---
## 6. Key Figures

In [ ]:
# ── Figure 1: R² bar chart with error bars ────────────────────────────────────
all_names = list(results_reg.keys()) + ['MLP (Lightning)']
all_means = [results_reg[n].loc['mean', 'R2'] for n in results_reg] + [mlp_mean.R2]
all_stds  = [results_reg[n].loc['std',  'R2'] for n in results_reg] + [mlp_std.R2]
colors    = ['steelblue', 'darkorange', 'green', 'purple']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(all_names, all_means, xerr=all_stds,
               color=colors, capsize=5, edgecolor='white', height=0.5)
ax.axvline(0.81, color='red', linestyle='--', linewidth=1.2,
           label='SVR literature (Vignaux 2023, AChE, R²=0.81)')
for bar, mean, std in zip(bars, all_means, all_stds):
    ax.text(mean + std + 0.005, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontsize=9)
ax.set_xlabel('R² (Butina 5-fold CV, ECFP6)')
ax.set_title('Model Comparison — pChEMBL Regression', fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0, 1.05)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('part5_r2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: R² per fold — show variance ─────────────────────────────────────
folds = [1, 2, 3, 4, 5]
fig, ax = plt.subplots(figsize=(9, 4))

all_dfs = list(results_reg.values()) + [mlp_df]
for name, res, col in zip(all_names, all_dfs, colors):
    vals = res.loc[folds, 'R2'].astype(float)
    ax.plot(folds, vals, 'o-', color=col, label=name, linewidth=1.5)
    ax.axhline(vals.mean(), color=col, linestyle='--', alpha=0.3)

ax.set_xlabel('Fold')
ax.set_ylabel('R²')
ax.set_title('R² Stability Across Butina CV Folds', fontweight='bold')
ax.set_xticks(folds)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('part5_r2_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()
print('High fold variance is expected with Butina CV — some clusters')
print('are harder to predict (structurally distinct from training data).')

---
## 7. Save Best Model

In [ ]:
all_r2 = {n: results_reg[n].loc['mean', 'R2'] for n in results_reg}
all_r2['MLP (Lightning)'] = mlp_mean.R2
best_name = max(all_r2, key=all_r2.get)
print(f'Best model: {best_name}  (mean R² = {all_r2[best_name]:.4f})')

if best_name != 'MLP (Lightning)':
    model_map = {'RF (tuned)': rf_tuned,
                 'XGBoost (tuned)': xgb_tuned,
                 'SVR (tuned)': svr_tuned}
    best_model = model_map[best_name]
    best_model.fit(X, y_reg)  # train on full dataset for Part 6 SHAP
    with open('best_model_part5.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print('Saved best_model_part5.pkl')
else:
    dm_full = EGFRDataModule(X, y_reg, X, y_reg, batch_size=64)
    best_mlp = MLPRegressor(input_dim=input_dim)
    L.Trainer(max_epochs=60, accelerator='auto', devices=1,
              enable_progress_bar=True, logger=False,
              enable_checkpointing=False).fit(best_mlp, dm_full)
    torch.save(best_mlp.state_dict(), 'best_mlp_part5.pt')
    print('Saved best_mlp_part5.pt')

np.save('X_for_shap.npy', X)   # feature matrix used for SHAP in Part 6
print('Saved X_for_shap.npy')

---
## Summary

### Files produced
| File | Used in |
|---|---|
| `part5_final_comparison.csv` | README benchmark table |
| `best_model_part5.pkl` | Part 6 SHAP + Part 7 applicability domain |
| `X_for_shap.npy` | Part 6 SHAP |
| `part5_r2_comparison.png` | README figure |
| `part5_r2_per_fold.png` | README figure |

### README-ready result line (fill in your numbers)
```
Best model: [XGBoost / SVR] trained on ECFP6 fingerprints
R² = X.XXX ± Y.YYY (Butina 5-fold CV, n ≈ 2400 EGFR compounds)
Comparable to RF/SVR baselines in Vignaux et al. 2023 (AChE, R² = 0.81).
Direct comparison is limited by differences in target and split protocol.
```

### Next: Part 6 — SHAP Explainability
Loads `best_model_part5.pkl` + `X_maccs.csv` to produce:
- SHAP beeswarm plot (top 20 MACCS keys)
- Molecular similarity maps for Erlotinib and Osimertinib
- Biological interpretation: do high-SHAP bits match known EGFR pharmacophores?